# Integrated Gradients — extended multi-model bias analysis (`41_1`)

This notebook **extends** `41_integrated_gradients.ipynb`: same **Integrated Gradients** setup on **BERT embeddings**, matching bar / **heatmap** styling, but adds **several saved checkpoints**, **cross-model panels on identical résumés**, and explicit **city / age / gender-related** cue tracing.

**Constraints:** no retraining; no large hyperparameter or full-corpus attribution sweeps. All models share one **fixed random subset** of the test split for forward passes; Integrated Gradients runs on **curated rows** and on **small bounded scans** for aggregate heatmaps.

| Outputs | Path |
|---------|------|
| Tables | `notebooks/results/integrated_gradients_extended/` |
| Figures | `figures/integrated_gradients_extended/` |

**Blocks:** (A) baseline reference panels, (B) per–new-model bias-oriented panels, (C) cross-model comparison on shared indices, (D) proxy heatmaps and city-attribution mass, (E) key takeaways.


In [1]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from captum.attr import LayerIntegratedGradients
from datasets import Dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)

_cwd = Path.cwd().resolve()
if (_cwd / "data" / "processed").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "processed").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise SystemExit("Run from repository root or notebooks/ (need data/processed).")

PROCESSED = REPO_ROOT / "data" / "processed"
MODELS_ROOT = REPO_ROOT / "notebooks" / "models"
RESULTS_DIR = REPO_ROOT / "notebooks" / "results" / "integrated_gradients_extended"
FIGS_DIR = REPO_ROOT / "figures" / "integrated_gradients_extended"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print("REPO_ROOT:", REPO_ROOT)
print("Device:", device)


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
Device: mps


In [2]:
PROJECT_TAG = "integrated_gradients_extended"
SUBSET_SEED = 41
SUBSET_MAX = 2800
BATCH = 32
MAX_LEN = 128
IG_STEPS = 18
TOP_K = 15

MODEL_SPECS = [
    ("baseline", "bert_9classes_final", "Baseline"),
    ("scrubbing", "bert_scrubbing", "Data scrubbing"),
    ("gdro", "bert_gdro_eta01_2ep", "GroupDRO η=0.1"),
    ("focal", "bert_focal_loss", "Focal loss"),
    ("label_smooth", "bert_label_smoothing", "Label smoothing"),
    ("debiased", "bert_debiased_combo", "Debiased combo"),
    ("oversample", "bert_oversample_only", "Oversample only"),
]

manifest = {
    "project": PROJECT_TAG,
    "device": str(device),
    "subset_seed": SUBSET_SEED,
    "subset_max": SUBSET_MAX,
    "ig_steps": IG_STEPS,
    "models": [{"slug": s[0], "path": str(MODELS_ROOT / s[1]), "title": s[2]} for s in MODEL_SPECS],
}

missing = [sub for _, sub, _ in MODEL_SPECS if not (MODELS_ROOT / sub).is_dir()]
assert not missing, f"Missing model dirs: {missing}"
print("Models:", len(MODEL_SPECS))


Models: 7


In [3]:
df_test = pd.read_csv(PROCESSED / "test.csv")
mapping_df = pd.read_csv(PROCESSED / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping_df["label"], mapping_df["supercategory"]))
df_test["supercategory"] = df_test["label"].map(label_to_supercat)

_le0 = joblib.load(MODELS_ROOT / MODEL_SPECS[0][1] / "label_encoder.joblib")
df_test["y_true"] = _le0.transform(df_test["supercategory"])
_CLASS_IDX = {name: i for i, name in enumerate(_le0.classes_)}
id2label = dict(enumerate(_le0.classes_))

rng = np.random.default_rng(SUBSET_SEED)
n_sub = min(SUBSET_MAX, len(df_test))
subset_ix = np.sort(rng.choice(df_test.index.values, size=n_sub, replace=False))
df_sub = df_test.loc[subset_ix].copy()
manifest["subset_n"] = int(len(subset_ix))
print("Test rows:", len(df_test), "| Subset:", len(df_sub))


Test rows: 5510 | Subset: 2800


## Forward pass on the shared subset


In [4]:
def load_bundle(model_dir: Path):
    tok = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), local_files_only=True)
    model = model.to(device).eval()
    le = joblib.load(model_dir / "label_encoder.joblib")
    return {"dir": model_dir, "tokenizer": tok, "model": model, "le": le}


def predict_on_df(bundle, df: pd.DataFrame) -> np.ndarray:
    tok, model = bundle["tokenizer"], bundle["model"]

    def tok_fn(batch):
        return tok(batch["resume_text"], padding="max_length", truncation=True, max_length=MAX_LEN)

    ds = Dataset.from_pandas(df[["resume_text"]])
    ds = ds.map(tok_fn, batched=True)
    ds.set_format("torch", columns=["input_ids", "attention_mask"])
    loader = DataLoader(ds, batch_size=BATCH)
    preds = []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            preds.append(out.logits.argmax(-1).cpu().numpy())
    return np.concatenate(preds, axis=0)


bundles: dict[str, dict] = {}
for slug, sub, _ in MODEL_SPECS:
    bundles[slug] = load_bundle(MODELS_ROOT / sub)

for slug, _, _ in tqdm(MODEL_SPECS, desc="Subset predict"):
    y = predict_on_df(bundles[slug], df_sub)
    df_sub[f"pred__{slug}"] = y
    df_sub[f"ok__{slug}"] = df_sub["y_true"].values == y

wide = df_sub[
    ["resume_text", "supercategory", "label", "city_group", "gender", "age_group", "y_true"]
].copy()
for slug, _, _ in MODEL_SPECS:
    wide[f"pred__{slug}"] = df_sub[f"pred__{slug}"]
    wide[f"ok__{slug}"] = df_sub[f"ok__{slug}"]
wide.insert(0, "df_index", df_sub.index)
wide.to_csv(RESULTS_DIR / "01_subset_predictions_wide.csv", index=False)
print("Saved", RESULTS_DIR / "01_subset_predictions_wide.csv")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/201 [00:00<00:00, 9258.95it/s, Materializing param=bert.embeddings.LayerNorm.bias]

Loading weights:   0%|          | 1/201 [00:00<00:00, 951.52it/s, Materializing param=bert.embeddings.LayerNorm.bias] 

Loading weights:   1%|          | 2/201 [00:00<00:00, 434.42it/s, Materializing param=bert.embeddings.LayerNorm.weight]

Loading weights:   1%|          | 2/201 [00:00<00:00, 302.98it/s, Materializing param=bert.embeddings.LayerNorm.weight]

...

Map: 100%|██████████| 2800/2800 [00:03<00:00, 750.32 examples/s]

Map: 100%|██████████| 2800/2800 [00:03<00:00, 720.37 examples/s]

Subset predict: 100%|██████████| 7/7 [12:15<00:00, 101.03s/it]

Subset predict: 100%|██████████| 7/7 [12:15<00:00, 105.07s/it]

Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/integrated_gradients_extended/01_subset_predictions_wide.csv


## IG helpers (same spirit as `41`)


In [5]:
def make_lig(model):
    def forward_fn(input_ids, attention_mask):
        return model(input_ids=input_ids, attention_mask=attention_mask).logits

    return LayerIntegratedGradients(forward_fn, model.bert.embeddings)


def compute_ig(lig, tok, model, text: str, target_class: int, n_steps: int = IG_STEPS):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding="max_length")
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    attributions = lig.attribute(
        inputs=input_ids,
        additional_forward_args=(attention_mask,),
        target=int(target_class),
        n_steps=n_steps,
    )
    scores = attributions.sum(dim=-1).squeeze().detach().cpu().numpy()
    tok_list = tok.convert_ids_to_tokens(input_ids[0].cpu())
    am = attention_mask[0].cpu().numpy()
    return [(t, s) for t, s, m in zip(tok_list, scores, am) if m == 1]


def merge_wordpieces(token_scores):
    merged = []
    cur_w, cur_s = "", 0.0
    for t, s in token_scores:
        if t.startswith("##"):
            cur_w += t[2:]
            cur_s += s
        else:
            if cur_w:
                merged.append((cur_w, cur_s))
            cur_w, cur_s = t, s
    if cur_w:
        merged.append((cur_w, cur_s))
    return merged


def clean_filter(word_scores, min_len=3):
    special = {"[CLS]", "[SEP]", "[MASK]"}
    year_re = re.compile(r"^\d{4}$")
    garbage_re = re.compile(r"^[\W\d_]+$")
    stops = {"the", "and", "for", "with", "from", "that", "this", "are", "was", "were", "have", "has", "been"}
    out = []
    for w, s in word_scores:
        if w in special or year_re.match(w) or garbage_re.match(w) or len(w) < min_len:
            continue
        if w.lower() in stops:
            continue
        out.append((w, s))
    return out


def get_top_attr(lig, tok, model, text: str, target_class: int, top_k: int = TOP_K):
    raw = compute_ig(lig, tok, model, text, target_class)
    merged = merge_wordpieces(raw)
    cleaned = clean_filter(merged)
    return sorted(cleaned, key=lambda x: abs(x[1]), reverse=True)[:top_k]


def plot_bar(word_scores, title, ax):
    if not word_scores:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=11)
        ax.set_title(title, fontsize=8, fontweight="bold")
        return
    words = [w for w, _ in word_scores]
    scores = [s for _, s in word_scores]
    colors = ["#2ecc71" if s > 0 else "#e74c3c" for s in scores]
    y_pos = np.arange(len(words))
    ax.barh(y_pos, scores, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(words, fontsize=7)
    ax.invert_yaxis()
    ax.axvline(0, color="black", lw=0.5)
    ax.set_title(title, fontsize=8, fontweight="bold")


def extract_word_attr_map(lig, tok, model, text: str, target_class: int, vocab: list[str]):
    raw = compute_ig(lig, tok, model, text, target_class)
    merged = merge_wordpieces(raw)
    hits: dict[str, list[float]] = {}
    for w, s in merged:
        wl = w.lower()
        for key in vocab:
            if key in wl or wl in key:
                hits.setdefault(key, []).append(float(s))
    return hits


def title_for_row(r, note: str, pred_slug: str):
    yt = id2label[int(r["y_true"])]
    yp = id2label[int(r[f"pred__{pred_slug}"])]
    ok = bool(r[f"ok__{pred_slug}"])
    meta = ", ".join(
        filter(
            None,
            [str(r.get("city_group", "")), str(r.get("gender", "")), str(r.get("age_group", ""))],
        )
    )
    return f"{'OK' if ok else 'ERR'} {yt} → {yp}\n[{note}]\n{meta}"


lig_cache = {slug: make_lig(bundles[slug]["model"]) for slug, _, _ in MODEL_SPECS}
print("IG ready:", len(lig_cache), "models")


IG ready: 7 models


## (A) Baseline reference panels


In [6]:
used_panel = set()


def take_row(df: pd.DataFrame, mask: pd.Series):
    m = mask & ~df.index.isin(used_panel)
    if not m.any():
        return None
    r = df.loc[m].iloc[0]
    used_panel.add(r.name)
    return r


small_cities = ["Пермь", "Воронеж", "Омск", "Красноярск", "Томск", "Ижевск", "Тверь"]
bslug = "baseline"
dfp = df_sub

figure_configs = [
    {
        "name": "ig_baseline_panel_cities",
        "title": "Baseline: cities (large vs small hubs, subset)",
        "rows": [
            ("moscow_ok", (dfp["city_group"] == "Москва") & dfp["ok__baseline"]),
            ("moscow_err", (dfp["city_group"] == "Москва") & ~dfp["ok__baseline"]),
            ("small_ok", dfp["city_group"].isin(small_cities) & dfp["ok__baseline"]),
            ("small_err", dfp["city_group"].isin(small_cities) & ~dfp["ok__baseline"]),
        ],
    },
    {
        "name": "ig_baseline_panel_gender",
        "title": "Baseline: gender slices",
        "rows": [
            ("male_ok", (dfp["gender"] == "Male") & dfp["ok__baseline"]),
            ("male_err", (dfp["gender"] == "Male") & ~dfp["ok__baseline"]),
            ("female_ok", (dfp["gender"] == "Female") & dfp["ok__baseline"]),
            ("female_err", (dfp["gender"] == "Female") & ~dfp["ok__baseline"]),
        ],
    },
    {
        "name": "ig_baseline_panel_age",
        "title": "Baseline: age groups (error-focused)",
        "rows": [
            ("age_22_25_err", (dfp["age_group"] == "22–25") & ~dfp["ok__baseline"]),
            ("age_36_50_err", (dfp["age_group"] == "36–50") & ~dfp["ok__baseline"]),
            ("age_50p_err", (dfp["age_group"] == "50+") & ~dfp["ok__baseline"]),
            ("age_young_err", dfp["age_group"].isin(["<18", "18–21"]) & ~dfp["ok__baseline"]),
        ],
    },
    {
        "name": "ig_baseline_panel_professions",
        "title": "Baseline: profession misreads",
        "rows": [
            ("be_err", (dfp["y_true"] == _CLASS_IDX["backend_general_dev"]) & ~dfp["ok__baseline"]),
            ("fe_err", (dfp["y_true"] == _CLASS_IDX["web_frontend"]) & ~dfp["ok__baseline"]),
            ("sys_err", (dfp["y_true"] == _CLASS_IDX["sysadmin_devops_network"]) & ~dfp["ok__baseline"]),
            ("sup_err", (dfp["y_true"] == _CLASS_IDX["tech_support_helpdesk"]) & ~dfp["ok__baseline"]),
        ],
    },
    {
        "name": "ig_baseline_panel_regions",
        "title": "Baseline: regional hubs (errors)",
        "rows": [
            ("spb_err", (dfp["city_group"] == "Санкт-Петербург") & ~dfp["ok__baseline"]),
            ("ekb_err", (dfp["city_group"] == "Екатеринбург") & ~dfp["ok__baseline"]),
            ("kzn_err", (dfp["city_group"] == "Казань") & ~dfp["ok__baseline"]),
            ("other_err", (dfp["city_group"] == "Other") & ~dfp["ok__baseline"]),
        ],
    },
]

blig = lig_cache["baseline"]
btok = bundles["baseline"]["tokenizer"]
bmodel = bundles["baseline"]["model"]
panel_manifest = []

for fc in figure_configs:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, (tag, mask) in zip(fig.axes, fc["rows"]):
        r = take_row(dfp, mask)
        if r is None:
            ax.text(0.5, 0.5, f"No row: {tag}", ha="center", va="center")
            ax.set_title(tag)
            panel_manifest.append({"figure": fc["name"], "panel": tag, "df_index": None})
            continue
        tw = get_top_attr(blig, btok, bmodel, r["resume_text"], int(r["pred__baseline"]))
        plot_bar(tw, title_for_row(r, tag, "baseline"), ax)
        panel_manifest.append({"figure": fc["name"], "panel": tag, "df_index": int(r.name)})
    fig.suptitle(fc["title"], fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    outp = FIGS_DIR / f"{fc['name']}.png"
    fig.savefig(outp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", outp)

pd.DataFrame(panel_manifest).to_csv(RESULTS_DIR / "02_baseline_panels_index.csv", index=False)


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_baseline_panel_cities.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_baseline_panel_gender.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_baseline_panel_age.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_baseline_panel_professions.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_baseline_panel_regions.png


## (B) New-model panels (bias-oriented four-panel)


In [7]:
city_lex = [
    "москва",
    "петербург",
    "спб",
    "казань",
    "екатеринбург",
    "новосибирск",
    "нижний",
    "краснодар",
    "самара",
    "ростов",
    "уфа",
    "воронеж",
    "пермь",
    "омск",
]
age_hint_re = re.compile(
    r"(лет|года|возраст|пенси|студент|выпускник|\b22\b|\b25\b|\b36\b|\b50\b|\b60\b)",
    re.I,
)

meta_city_toponym_in_body = []
for idx, row in df_sub.iterrows():
    cg = str(row.get("city_group", "") or "")
    if not cg or cg.lower() == "other":
        continue
    norm_c = cg.lower().replace("ё", "е")
    txt = str(row["resume_text"]).lower().replace("ё", "е")
    if norm_c in txt:
        meta_city_toponym_in_body.append(idx)

new_slugs = [s[0] for s in MODEL_SPECS if s[0] != "baseline"]
nm_manifest = []

for slug in tqdm(new_slugs, desc="New-model 4-panels"):
    used: set = set()

    def pick(mask: pd.Series):
        m = mask & ~df_sub.index.isin(used)
        if not m.any():
            return None
        row = df_sub.loc[m].iloc[0]
        used.add(row.name)
        return row

    p_moscow = pick((df_sub["city_group"] == "Москва") & ~df_sub[f"ok__{slug}"])
    p_female = pick(df_sub["gender"].astype(str).str.lower().eq("female") & ~df_sub[f"ok__{slug}"])
    if p_female is None:
        p_female = pick(df_sub["gender"].astype(str).str.lower().eq("female"))

    pool_c = [i for i in meta_city_toponym_in_body if i not in used]
    p_city = None
    if pool_c:
        ix = int(rng.choice(pool_c))
        used.add(ix)
        p_city = df_sub.loc[ix]

    pool_a = [i for i in df_sub.index if i not in used and age_hint_re.search(str(df_sub.loc[i, "resume_text"]))]
    p_age = None
    if pool_a:
        ix = int(rng.choice(pool_a))
        p_age = df_sub.loc[ix]

    quad = [
        ("moscow_error", p_moscow, "Moscow / model error"),
        ("female_slice", p_female, "Female slice"),
        ("city_metadata_echo", p_city, "Metadata city also in body"),
        ("age_surface_form", p_age, "Age / experience wording"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, (tag, prow, note) in zip(fig.axes, quad):
        if prow is None:
            ax.text(0.5, 0.5, f"No row: {tag}", ha="center", va="center")
            ax.set_title(tag)
            nm_manifest.append({"model": slug, "panel": tag, "df_index": None})
            continue
        tw = get_top_attr(lig_cache[slug], bundles[slug]["tokenizer"], bundles[slug]["model"], prow["resume_text"], int(prow[f"pred__{slug}"]))
        plot_bar(tw, title_for_row(prow, note, slug), ax)
        nm_manifest.append({"model": slug, "panel": tag, "df_index": int(prow.name)})
    fig.suptitle(f"{slug}: challenger — four bias-facing slices", fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    outp = FIGS_DIR / f"ig_model_panel_{slug}.png"
    fig.savefig(outp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", outp)

pd.DataFrame(nm_manifest).to_csv(RESULTS_DIR / "03_new_model_panels_index.csv", index=False)


New-model 4-panels:   0%|          | 0/6 [00:00<?, ?it/s]

New-model 4-panels:  17%|█▋        | 1/6 [00:08<00:40,  8.00s/it]

Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_model_panel_scrubbing.png


New-model 4-panels:  33%|███▎      | 2/6 [00:18<00:38,  9.63s/it]

Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_model_panel_gdro.png


...

New-model 4-panels:  83%|████████▎ | 5/6 [01:26<00:19, 19.20s/it]

Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_model_panel_debiased.png


New-model 4-panels: 100%|██████████| 6/6 [01:45<00:00, 19.02s/it]

New-model 4-panels: 100%|██████████| 6/6 [01:45<00:00, 17.50s/it]

Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_model_panel_oversample.png


## (C) Shared-index cross-model comparison


In [8]:
def fine_label_echo_mask(df: pd.DataFrame) -> pd.Series:
    def row_ok(row) -> bool:
        fl = str(row.get("label", "") or "").strip().lower()
        if len(fl) < 4:
            return False
        return fl in str(row.get("resume_text", "") or "").lower()

    return df.apply(row_ok, axis=1)


fairness_rows = []
p_fair = REPO_ROOT / "notebooks" / "results" / "fairness_error_analysis" / "05_representative_slice_excerpts_gt_only.csv"
if p_fair.is_file():
    ff = pd.read_csv(p_fair)
    if "df_index" in ff.columns:
        fairness_rows = [int(x) for x in ff["df_index"].dropna().unique()[:3] if int(x) in df_sub.index]

compare_indices: list[tuple[int, str]] = []
seen_ci: set[int] = set()

def add_ci(ix: int, note: str):
    if ix in df_sub.index and ix not in seen_ci:
        compare_indices.append((ix, note))
        seen_ci.add(ix)


# Proxy / leakage style
m_echo = fine_label_echo_mask(df_sub)
if m_echo.any():
    add_ci(int(df_sub.loc[m_echo].index[0]), "fine_label_substring_in_body")

for ix in meta_city_toponym_in_body[:30]:
    add_ci(int(ix), "city_group_toponym_in_text")

# Female slice
pool_f = df_sub[df_sub["gender"].astype(str).str.lower().eq("female")].index.tolist()
if pool_f:
    add_ci(int(rng.choice(pool_f)), "female_metadata")

# Baseline error with city lex in text
cand = df_sub[df_sub["resume_text"].str.lower().apply(lambda t: any(c in t for c in city_lex)) & ~df_sub["ok__baseline"]]
if len(cand):
    add_ci(int(cand.index[0]), "city_lex_in_text_baseline_error")

# Model disagreement: baseline vs GroupDRO
dis = df_sub[df_sub["pred__baseline"] != df_sub["pred__gdro"]]
if len(dis):
    add_ci(int(dis.index[0]), "baseline_vs_gdro_disagreement")

for ix in fairness_rows:
    add_ci(ix, "fairness_slice_csv")

compare_indices = compare_indices[:8]
rows_records = []
for ix, note in compare_indices:
    r = df_sub.loc[ix]
    rows_records.append({"df_index": ix, "category": note})
pd.DataFrame(rows_records).to_csv(RESULTS_DIR / "04_compare_row_manifest.csv", index=False)

# Tall stacked figure per shared résumé
for ix, note in compare_indices:
    r = df_sub.loc[ix]
    n_m = len(MODEL_SPECS)
    fig, axes = plt.subplots(n_m, 1, figsize=(11, 2.2 * n_m), sharex=False)
    if n_m == 1:
        axes = [axes]
    for ax, (slug, _, mt) in zip(axes, MODEL_SPECS):
        tw = get_top_attr(lig_cache[slug], bundles[slug]["tokenizer"], bundles[slug]["model"], r["resume_text"], int(r[f"pred__{slug}"]))
        plot_bar(tw, f"{mt}\n" + title_for_row(r, note, slug), ax)
    fig.suptitle(f"Shared résumé df_index={ix} — {note}", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.99])
    safe = re.sub(r"[^a-z0-9]+", "_", note.lower()).strip("_")[:40]
    outp = FIGS_DIR / f"ig_compare_stack_{ix}_{safe}.png"
    fig.savefig(outp, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", outp)

# Token overlap cosine: top attributed words (by |IG|), intersection alignment only
def attr_dict(lig, tok, model, text: str, pred_c: int, cap=48):
    raw = compute_ig(lig, tok, model, text, pred_c)
    merged = merge_wordpieces(raw)
    merged = sorted(merged, key=lambda x: abs(x[1]), reverse=True)[:cap]
    return {w.lower(): float(s) for w, s in merged}


def cos_on_overlap(d1: dict, d2: dict):
    keys = set(d1) & set(d2)
    if len(keys) < 3:
        return float("nan")
    v1 = np.array([d1[k] for k in keys], dtype=np.float32)
    v2 = np.array([d2[k] for k in keys], dtype=np.float32)
    denom = (np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-12
    return float((v1 * v2).sum() / denom)


sim_rows = []
ref_slug = "baseline"
for ix, note in compare_indices:
    r = df_sub.loc[ix]
    d0 = attr_dict(lig_cache[ref_slug], bundles[ref_slug]["tokenizer"], bundles[ref_slug]["model"], r["resume_text"], int(r[f"pred__{ref_slug}"]))
    for slug, _, title in MODEL_SPECS:
        if slug == ref_slug:
            continue
        d1 = attr_dict(lig_cache[slug], bundles[slug]["tokenizer"], bundles[slug]["model"], r["resume_text"], int(r[f"pred__{slug}"]))
        sim_rows.append(
            {
                "df_index": ix,
                "compare_note": note,
                "model": slug,
                "cos_vs_baseline_intersection": cos_on_overlap(d0, d1),
                "overlap_n": len(set(d0) & set(d1)),
            }
        )

pd.DataFrame(sim_rows).to_csv(RESULTS_DIR / "05_compare_attribution_similarity.csv", index=False)


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_0_fine_label_substring_in_body.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_1_city_group_toponym_in_text.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_2_city_group_toponym_in_text.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_5_city_group_toponym_in_text.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_13_city_group_toponym_in_text.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_22_city_group_toponym_in_text.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_23_city_group_toponym_in_text.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/ig_compare_stack_24_city_group_toponym_in_text.png


## (D) Heatmaps & proxy mass (city / gender / age)


In [9]:
# Word lists (Russian surface forms — same family as notebook 41)
city_words = [
    "москва",
    "петербург",
    "спб",
    "казань",
    "екатеринбург",
    "новосибирск",
    "магнитогорск",
    "ижевск",
    "нижний",
    "ростов",
    "красноярск",
    "краснодар",
    "пермь",
    "самара",
    "уфа",
    "челябинск",
    "воронеж",
    "омск",
    "томск",
]
gender_words = ["мужчина", "женщина", "муж", "жена", "отец", "мать", "армия", "декрет"]
age_words = ["лет", "года", "возраст", "пенси", "студент", "выпускник", "молод", "18", "21", "25", "36", "50"]


def collect_proxy_matrix(df_sample: pd.DataFrame, slug: str, vocab: list[str], prof_col_data: dict):
    # Mutate prof_col_data[prof_name][word] -> list of scores.
    lig_ = lig_cache[slug]
    tok_ = bundles[slug]["tokenizer"]
    mod_ = bundles[slug]["model"]
    for _, row in df_sample.iterrows():
        y_tid = int(row["y_true"])
        prof = id2label[y_tid]
        pred_c = int(row[f"pred__{slug}"])
        hits = extract_word_attr_map(lig_, tok_, mod_, row["resume_text"], pred_c, vocab)
        for w, scores in hits.items():
            prof_col_data.setdefault(prof, {}).setdefault(w, [])
            prof_col_data[prof][w].extend(scores)


def heatmap_from_nested(nested: dict[str, dict[str, list[float]]], y_words: list[str], title: str, fname: str):
    profs = list(id2label.values())
    mat = np.zeros((len(y_words), len(profs)))
    for i, w in enumerate(y_words):
        for j, p in enumerate(profs):
            s = nested.get(p, {}).get(w, [])
            mat[i, j] = float(np.mean(s)) if s else 0.0
    fig, ax = plt.subplots(figsize=(14, max(4.0, 0.45 * len(y_words))))
    im = ax.imshow(mat, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(profs)))
    ax.set_xticklabels([p.replace("_", "\n") for p in profs], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(y_words)))
    ax.set_yticklabels(y_words, fontsize=9)
    for i in range(len(y_words)):
        for j in range(len(profs)):
            v = mat[i, j]
            if v != 0:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if abs(v) > 0.05 else "black")
    plt.colorbar(im, ax=ax, label="Mean IG (matched span)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGS_DIR / fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", FIGS_DIR / fname)


# Bounded scan: at most 2 rows per true class with city lex in body
scan_rows = []
for tid in sorted(id2label):
    sub = df_sub[(df_sub["y_true"] == tid) & df_sub["resume_text"].str.lower().apply(lambda t: any(c in t for c in city_lex))]
    if len(sub):
        scan_rows.append(sub.head(2))
scan_df = pd.concat(scan_rows, axis=0).drop_duplicates() if scan_rows else df_sub.iloc[:0]
if len(scan_df) < 8:
    extra = df_sub[df_sub["resume_text"].str.lower().apply(lambda t: any(c in t for c in city_lex))]
    scan_df = pd.concat([scan_df, extra.head(18)], axis=0).drop_duplicates()
print("Heatmap scan rows (city-triggered):", len(scan_df))

for slug, _, mt in tqdm(MODEL_SPECS, desc="Per-model proxy heatmaps"):
    city_prof: dict = {}
    collect_proxy_matrix(scan_df, slug, city_words, city_prof)
    active_cities = [w for w in city_words if any(city_prof.get(p, {}).get(w) for p in id2label.values())]
    if not active_cities:
        active_cities = city_words[:12]
    heatmap_from_nested(city_prof, active_cities, f"{mt}: city-linked tokens × profession (subset scan)", f"heatmap_city_{slug}.png")

    g_prof: dict = {}
    g_scan = df_sub[df_sub["resume_text"].str.lower().apply(lambda t: any(g in t for g in gender_words))].head(40)
    collect_proxy_matrix(g_scan, slug, gender_words, g_prof)
    active_g = [w for w in gender_words if any(g_prof.get(p, {}).get(w) for p in id2label.values())]
    if active_g:
        heatmap_from_nested(g_prof, active_g, f"{mt}: gender-related tokens × profession", f"heatmap_gender_{slug}.png")

    a_prof: dict = {}
    a_scan = df_sub[df_sub["resume_text"].str.lower().apply(lambda t: any(a in t for a in age_words))].head(40)
    collect_proxy_matrix(a_scan, slug, age_words, a_prof)
    active_a = [w for w in age_words if any(a_prof.get(p, {}).get(w) for p in id2label.values())]
    if active_a:
        heatmap_from_nested(a_prof, active_a, f"{mt}: age-related tokens × profession", f"heatmap_age_{slug}.png")

# City attribution mass ratio (shared texts): sum |IG| on city_lex tokens / sum |IG| all merged tokens
mass_records = []
shared_text_df = df_sub[df_sub["resume_text"].str.lower().apply(lambda t: any(c in t for c in city_lex))].head(35)

for _, row in tqdm(shared_text_df.iterrows(), total=len(shared_text_df), desc="City mass ratio"):
    ix = row.name
    for slug, _, mt in MODEL_SPECS:
        lig_ = lig_cache[slug]
        tok_ = bundles[slug]["tokenizer"]
        mod_ = bundles[slug]["model"]
        pred_c = int(row[f"pred__{slug}"])
        raw = compute_ig(lig_, tok_, mod_, row["resume_text"], pred_c)
        merged = merge_wordpieces(raw)
        total = sum(abs(s) for _, s in merged) + 1e-12
        city_sum = 0.0
        for w, s in merged:
            wl = w.lower()
            if any(c in wl for c in city_lex):
                city_sum += abs(s)
        mass_records.append(
            {"df_index": ix, "model": slug, "city_attr_mass_ratio": float(city_sum / total), "pred_supercat": id2label[pred_c]}
        )

mass_df = pd.DataFrame(mass_records)
mass_df.to_csv(RESULTS_DIR / "06_city_attr_mass_ratio_long.csv", index=False)
summ = mass_df.groupby("model")["city_attr_mass_ratio"].agg(["mean", "median", "max"]).reset_index()
summ.to_csv(RESULTS_DIR / "07_city_attr_mass_ratio_by_model.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
order = [s[0] for s in MODEL_SPECS]
sns.barplot(data=mass_df, x="model", y="city_attr_mass_ratio", order=order, ax=ax)
ax.set_title("|IG| on city-like merged tokens / total |IG|, shared résumés (same texts across models)")
ax.set_xlabel(None)
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIGS_DIR / "bar_city_attr_mass_by_model.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved summary heatmaps / mass bar")

with open(RESULTS_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)


Heatmap scan rows (city-triggered): 18


Per-model proxy heatmaps:   0%|          | 0/7 [00:00<?, ?it/s]

Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/heatmap_city_baseline.png


Saved /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/integrated_gradients_extended/heatmap_gender_baseline.png


Per-model proxy heatmaps:  14%|█▍        | 1/7 [05:49<34:56, 349.34s/it]

...

City mass ratio:  94%|█████████▍| 33/35 [16:10<00:07,  3.68s/it]

City mass ratio:  97%|█████████▋| 34/35 [16:14<00:03,  3.66s/it]

City mass ratio: 100%|██████████| 35/35 [16:18<00:00,  3.69s/it]

City mass ratio: 100%|██████████| 35/35 [16:18<00:00, 27.95s/it]

Saved summary heatmaps / mass bar


## Key takeaways (for the paper)

- **What IG adds:** Aggregate fairness tables and city-swap stress tests show *whether* decisions shift across groups or city rewrites; IG shows *which surface spans* participate in the decision for concrete résumés — qualification phrasing versus **toponyms**, **demographic** phrasing, template **label echoes**, or generic work-history tokens.
- **Cross-model panels:** When the **same text** is scored by the baseline, robustness (GroupDRO), and preprocessing (scrubbing / debiased) checkpoints, attribution stacks make it visible if an intervention **redistributes mass** off city/header tokens without a matching shift in predicted class — a qualitative counterpart to robust gap metrics.
- **Proxy-sensitive behavior:** Models with **higher mean city-|IG| mass** on shared city-bearing résumés are the strongest qualitative candidates for **geo-proxy** use, even when accuracy is similar. **Gender- and age-word** heatmaps are sparse (honest reporting): when tokens appear, compare whether **in-processing** methods shrink their mean signed IG versus baseline on the same scanned rows.
- **Linking to `40_*`:** Rows pulled from `fairness_error_analysis` excerpts anchor high-gap stories in **token-level** evidence; combine with **city-swap** artifacts to argue whether shifts are tied to **rewritable** location fields versus stable skill phrases.

_Re-run time is bounded by subset size (`SUBSET_MAX`), the number of shared comparison rows, and per-model heatmap scans — adjust constants at the top of the specs cell if you need slightly denser coverage._
